In [1]:
import whisper
import librosa
import numpy as np
import soundfile as sf
import torch
from time import perf_counter
import warnings


https://www.kaggle.com/datasets/bryanpark/russian-single-speaker-speech-dataset?select=transcript.txt

# 1. Выбор модели 

Берём openai/whisper-tiny и openai/whisper-small

# 2. Распознавание для обеих моделей

Настоящий текст: За столицей мудрого царя Соломона шелестел по склонам холмов густой лес. С его опушки запутанные тропинки вели на поляну

In [2]:
model = whisper.load_model("tiny", device="cuda")
result = model.transcribe("clean.wav", language=None, word_timestamps=True,  fp16=False,)

print("Язык:", result["language"])
print("Текст:", result["text"].strip())

/home/topi768/FEFU7_remote/Нейронные сети/venv/lib/python3.13/site-packages/whisper/timing.py:42: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower median kernel implementation...
  warnings.warn(
/home/topi768/FEFU7_remote/Нейронные сети/venv/lib/python3.13/site-packages/whisper/timing.py:146: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower DTW implementation...
  warnings.warn(


Язык: ru
Текст: Засталиться и мудроваться рядом с ламона шлистел по склонам холмов в бустой лес. С его опушки запутанные тропинки вели на поляну.


In [3]:
model = whisper.load_model("small", device="cuda")
result = model.transcribe("clean.wav", language=None, word_timestamps=True, fp16=False)

print("Язык:", result["language"])
print("Текст:", result["text"].strip())

Язык: ru
Текст: За столицей мудрого царя Саламона шлистел по скалонам холмов густой лес. С его опушки запутанные тропинки вели на поляну.


/home/topi768/FEFU7_remote/Нейронные сети/venv/lib/python3.13/site-packages/whisper/timing.py:42: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower median kernel implementation...
  warnings.warn(
/home/topi768/FEFU7_remote/Нейронные сети/venv/lib/python3.13/site-packages/whisper/timing.py:146: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower DTW implementation...
  warnings.warn(


# 3. Прогон моделей на записи с шумом

In [4]:
y, sr = librosa.load('clean.wav', sr=None) # sr сохраняет исходную частоту


rng = np.random.default_rng(42)
noise_level = 0.1 * np.sqrt(np.mean(y**2)) # Белый шум со среднеквадратичной амплитудой около 10% от амплитуды сигнала
y_noisy = y + rng.normal(0, noise_level, size=y.shape)

sf.write("noisy.wav", y_noisy, sr)

/home/topi768/FEFU7_remote/Нейронные сети/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
model = whisper.load_model("tiny", device="cuda")
result = model.transcribe("noisy.wav", language=None, word_timestamps=True,  fp16=False,)

print("Язык:", result["language"])
print("Текст:", result["text"].strip())

Язык: bg
Текст: За столица и мудрого царя саламона шлистел по скалоном халмова го стои лес. Съява опушки запутанна е трапинки вели на палеяно.


/home/topi768/FEFU7_remote/Нейронные сети/venv/lib/python3.13/site-packages/whisper/timing.py:42: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower median kernel implementation...
  warnings.warn(
/home/topi768/FEFU7_remote/Нейронные сети/venv/lib/python3.13/site-packages/whisper/timing.py:146: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower DTW implementation...
  warnings.warn(


In [6]:
model = whisper.load_model("small", device="cuda")
result = model.transcribe("noisy.wav", language=None, word_timestamps=True, fp16=False)

print("Язык:", result["language"])
print("Текст:", result["text"].strip())

Язык: ru
Текст: За столицей мудрого царя Соломона шлистел по скалонам холмов густой лес. С его опушки запутанные тропинки вели на поляну.


/home/topi768/FEFU7_remote/Нейронные сети/venv/lib/python3.13/site-packages/whisper/timing.py:42: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower median kernel implementation...
  warnings.warn(
/home/topi768/FEFU7_remote/Нейронные сети/venv/lib/python3.13/site-packages/whisper/timing.py:146: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower DTW implementation...
  warnings.warn(


# 4. Замеры времени

In [7]:


warnings.filterwarnings(
    "ignore",
    message="Failed to launch Triton kernels",
    category=UserWarning,
    module=r"whisper\.timing",
)

In [8]:
for model_name in ("tiny", "small"):
    for audio_file in ("clean.wav", "noisy.wav"):
        start = perf_counter()
        model = whisper.load_model(model_name, device="cuda")
        torch.cuda.synchronize() # ждет пока видеокарта не закончит работу. пайтон омжет дойти до следующей тсроки не дождавшись ее работы
        load_time = perf_counter() - start


        # Один прогревочный прогон. Подготовка вычислений
        model.transcribe(audio_file, language=None, word_timestamps=True, fp16=False)
        torch.cuda.synchronize()

        start = perf_counter()
        model.transcribe(audio_file, language=None, word_timestamps=True, fp16=False)
        torch.cuda.synchronize()
        inference_time = perf_counter() - start

        print(f"{model_name}, {audio_file} - загрузка {load_time:.3f} с, "
              f"распознавание {inference_time:.3f} с")

        del model # очищает видеопамять. тк при загурзке новой модели в моменте в виддеопамяти будет хранится обе модели, что плохо

tiny, clean.wav - загрузка 0.451 с, распознавание 0.338 с
tiny, noisy.wav - загрузка 0.446 с, распознавание 0.365 с
small, clean.wav - загрузка 2.607 с, распознавание 1.684 с
small, noisy.wav - загрузка 2.594 с, распознавание 1.695 с


# 5. Оценка качества

оценка субъективная

**Настоящий текст** | За столицей мудрого царя Соломона шелестел по склонам холмов густой лес. С его опушки запутанные тропинки вели на поляну

**tiny, clean.wav** | Засталиться и мудроваться рядом с ламона шлистел по склонам холмов в бустой лес. С его опушки запутанные тропинки вели на поляну.

**small, clean.wav** |  За столицей мудрого царя Саламона шлистел по скалонам холмов густой лес. С его опушки запутанные тропинки вели на поляну

**tiny, noisy.wav** | За столица и мудрого царя саламона шлистел по скалоном халмова го стои лес. Съява опушки запутанна е трапинки вели на палеяно

**small, noisy.wav** | За столицей мудрого царя Соломона шлистел по скалонам холмов густой лес. С его опушки запутанные тропинки вели на поляну.


In [9]:
reference = (
    "За столицей мудрого царя Соломона шелестел по склонам холмов густой лес. "
    "С его опушки запутанные тропинки вели на поляну"
)

texts = {
    "tiny, clean.wav": (
        "Засталиться и мудроваться рядом с ламона шлистел по склонам холмов "
        "в бустой лес. С его опушки запутанные тропинки вели на поляну."
    ),
    "small, clean.wav": (
        "За столицей мудрого царя Саламона шлистел по скалонам холмов густой лес. "
        "С его опушки запутанные тропинки вели на поляну"
    ),
    "tiny, noisy.wav": (
        "За столица и мудрого царя саламона шлистел по скалоном халмова го стои лес. "
        "Съява опушки запутанна е трапинки вели на палеяно"
    ),
    "small, noisy.wav": (
        "За столицей мудрого царя Соломона шлистел по скалонам холмов густой лес. "
        "С его опушки запутанные тропинки вели на поляну."
    ),
}


In [10]:
def words(text):
    return text.lower().replace(".", "").split()

In [11]:

def count_errors(reference_words, recognized_words):
    n, m = len(reference_words), len(recognized_words)
    
    distance = [[0] * (m + 1) for _ in range(n + 1)]

    # Пустая строка: только вставки или удаления.
    for i in range(n + 1):
        distance[i][0] = i
    for j in range(m + 1):
        distance[0][j] = j

    for i, reference_word in enumerate(reference_words, start=1):
        for j, recognized_word in enumerate(recognized_words, start=1):
            deletion = distance[i - 1][j] + 1
            insertion = distance[i][j - 1] + 1
            substitution = distance[i - 1][j - 1] + (reference_word != recognized_word)
            distance[i][j] = min(deletion, insertion, substitution)

    return distance[n][m]



In [12]:

reference_words = words(reference)

for label, text in texts.items():
    errors = count_errors(reference_words, words(text))
    wer = errors / len(reference_words)
    print(f"{label} — ошибок {errors}, WER {wer:.2%}")

tiny, clean.wav — ошибок 9, WER 47.37%
small, clean.wav — ошибок 3, WER 15.79%
tiny, noisy.wav — ошибок 14, WER 73.68%
small, noisy.wav — ошибок 2, WER 10.53%



Перед расчётом все тексты приведены к нижнему регистру, точки удалены.
В эталонном тексте 19 слов. Для каждой расшифровки найдено минимальное
суммарное число замен, вставок и удалений слов.

$$
\mathrm{WER} = \frac{S + I + D}{N} \cdot 100\%
$$

Здесь $S$ — число замен, $I$ — число вставок, $D$ — число удалений,
$N$ — число слов в эталоне.

| Модель | Запись | Число ошибок | WER |
|---|---|---:|---:|
| `tiny` | Чистая | 9 | 47,37 % |
| `tiny` | Шумная | 14 | 73,68 % |
| `small` | Чистая | 3 | 15,79 % |
| `small` | Шумная | 2 | 10,53 % |



- **`tiny`, чистая запись.** Начало первого предложения сильно искажено.
  Например, "шелестел" распознано как "шлистел", а "густой" - как
  "в бустой". Второе предложение распознано полностью.
- **`tiny`, шумная запись.** Ошибки появились в обоих предложениях.
  Модель заменяла слова ("тропинки" в "трапинки", "поляну" в "палеяно")
  и ошибалась в их границах ("густой" в "го стои",
  "запутанные" - "запутанна е").
- **`small`, обе записи.** Смысл текста сохранён. На обеих записях повторились ошибки
  "шелестел" в "шлистел" и "склонам" в "скалонам". На чистой записи
  модель также распознала "Соломона" как "Саламона".

На данном фрагменте `small` показала меньший WER на обеих записях.
У `tiny` после добавления шума количество ошибок выросло с 9 до 14.
У `small` на шумной записи на одну ошибку меньше, поскольку в этом
прогоне модель правильно распознала слово "Соломона".

Вывод: модель tiny менее устойчивая к шуму, чем модель small

# 6. Длина распознанного текста (число символов) 

In [13]:
for label, text in texts.items():
    print(f"{label} — {len(text.strip())} символов")

tiny, clean.wav — 129 символов
small, clean.wav — 120 символов
tiny, noisy.wav — 125 символов
small, noisy.wav — 121 символов


# 7. Таблица результатов

| Модель | Тип записи | Загрузка, с | Инференс, с | Длина текста, символов | WER, % | Субъективная оценка качества |
|---|---|---:|---:|---:|---:|---|
| `tiny` | Чистая | 0,451 | 0,338 | 129 | 47,37 | Начало первого предложения сильно искажено. Второе предложение распознано полностью. |
| `tiny` | Шумная | 0,446 | 0,365 | 125 | 73,68 | Ошибки в обоих предложениях. Местами нарушены границы слов, текст трудно читать. |
| `small` | Чистая | 2,607 | 1,684 | 120 | 15,79 | Смысл сохранён. В первом предложении три ошибки, в том числе «Саламона» вместо «Соломона». |
| `small` | Шумная | 2,594 | 1,695 | 121 | 10,53 | Смысл сохранён. В первом предложении две ошибки — «шлистел» и «скалонам». |

# Дополнительное задание. Таймкоды сегментов/слов 

In [14]:
model = whisper.load_model("small", device="cuda")
result = model.transcribe("noisy.wav", language=None, word_timestamps=True, fp16=False)

print("Язык:", result["language"])
print("Текст:", result["text"].strip())

Язык: ru
Текст: За столицей мудрого царя Соломона шлистел по скалонам холмов густой лес. С его опушки запутанные тропинки вели на поляну.


In [18]:
result["segments"]

[{'id': 0,
  'seek': 0,
  'start': np.float64(0.0),
  'end': np.float64(4.78),
  'text': ' За столицей мудрого царя Соломона шлистел по скалонам холмов густой лес.',
  'tokens': [50364,
   22391,
   24053,
   27403,
   2345,
   1084,
   2955,
   481,
   2350,
   5188,
   2222,
   681,
   2933,
   1227,
   1253,
   14411,
   5941,
   1675,
   982,
   1414,
   2801,
   5239,
   1218,
   1784,
   1685,
   3490,
   1227,
   43632,
   2342,
   15028,
   1700,
   42548,
   13,
   50614],
  'temperature': 0.0,
  'avg_logprob': -0.18850126972904913,
  'compression_ratio': 1.510204081632653,
  'no_speech_prob': 0.008892793208360672,
  'words': [{'word': ' За',
    'start': np.float64(0.0),
    'end': np.float64(0.3),
    'probability': np.float64(0.941009521484375)},
   {'word': ' столицей',
    'start': np.float64(0.3),
    'end': np.float64(0.9),
    'probability': np.float64(0.9836554328600565)},
   {'word': ' мудрого',
    'start': np.float64(0.9),
    'end': np.float64(1.32),
    'probabil

In [15]:
for segment in result["segments"]:
    print(f"\n{segment['start']:.2f}–{segment['end']:.2f} с | {segment['text'].strip()}")

    for word in segment["words"]:
        print(f"  {word['start']:.2f}–{word['end']:.2f} с | {word['word'].strip()}")


0.00–4.78 с | За столицей мудрого царя Соломона шлистел по скалонам холмов густой лес.
  0.00–0.30 с | За
  0.30–0.90 с | столицей
  0.90–1.32 с | мудрого
  1.32–1.66 с | царя
  1.66–2.28 с | Соломона
  2.28–2.96 с | шлистел
  2.96–3.14 с | по
  3.14–3.62 с | скалонам
  3.62–4.00 с | холмов
  4.00–4.52 с | густой
  4.52–4.78 с | лес.

5.88–9.12 с | С его опушки запутанные тропинки вели на поляну.
  5.88–5.98 с | С
  5.98–6.16 с | его
  6.16–6.78 с | опушки
  6.78–7.70 с | запутанные
  7.70–8.26 с | тропинки
  8.26–8.50 с | вели
  8.50–8.64 с | на
  8.64–9.12 с | поляну.


Таймкоды сегментов можно использовать для создания субтитров, а таймкоды
слов - для перехода к нужному месту в записи